<a href="https://colab.research.google.com/github/andrearomano-collab/ML_oxidation_notebooks/blob/main/ML_oxidation_representative_time_profiles_four_features_FINAL.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Representative time profiles of four selected mass-spectral features

This standalone notebook reproduces the representative time-profile figure from the extracted DHS and DSI measurement CSV files and their `log.csv` files.

It deliberately contains only the calculations required for this figure:

1. read the log-referenced DHS and DSI measurement files;
2. average the four specified signals over the experiment-specific temporal windows;
3. normalise the signals to the summed acetone–ammonium reagent-ion signal near *m/z* 76 and 134;
4. subtract the matched normalised blank and replace negative results with zero;
5. normalise each feature to its maximum replicate-level intensity; and
6. plot the replicate means with 95% bootstrap confidence intervals.

The Wilcoxon/Bonferroni feature-screening stage is not repeated here because the identities of the four plotted features are prespecified. That screening determines whether a feature is retained but does not alter its normalised or background-subtracted intensity.


## Input data and configuration

Expected directory structure:

```text
DATA_ROOT/
├── DHS/
│   ├── log.csv
│   └── extracted DHS CSV files
└── DSI/
    ├── log.csv
    └── extracted DSI CSV files
```

Only measurement files named in the `File_Name` column of each log are read. In Google Colab, mount Google Drive if required and change `DATA_ROOT` accordingly. The default relative path works when the notebook and `source_data` directory are kept together in a cloned repository or supplementary-data package.


You can define the `DATA_ROOT` using the text box below. By default, it's set to `'source_data'`.

In [ ]:
from pathlib import Path

DATA_ROOT_STR = 'DATA_ROOT' # @param {type:"string"}
DATA_ROOT = Path(DATA_ROOT_STR)

print(f'Data root: {DATA_ROOT.resolve()}')

In [ ]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# List the contents of DATA_ROOT to verify the path
!ls -F "{DATA_ROOT}"
!ls -F "{DATA_ROOT / 'DHS'}"
!ls -F "{DATA_ROOT / 'DSI'}"

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

# Optional Google Colab setup:
# from google.colab import drive
# drive.mount('/content/drive')
# DATA_ROOT = Path('/content/drive/MyDrive/path/to/source_data')

OUTPUT_ROOT = DATA_ROOT / 'figure_output'
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

FIGURE_PATH = OUTPUT_ROOT / 'Representative_Time_Profiles_Normalized_to_Max_with_CI.jpg'

DHS_TARGETS = [130.122467]
DSI_TARGETS = [344.278442, 328.283234, 204.158478]

PLOT_ORDER = [
    'MZ_344.278442_DSI',
    'MZ_328.283234_DSI',
    'MZ_204.158478_DSI',
    'MZ_130.122467_DHS',
]

for required_path in [DATA_ROOT / 'DHS' / 'log.csv', DATA_ROOT / 'DSI' / 'log.csv']:
    if not required_path.exists():
        raise FileNotFoundError(
            f'Required file not found: {required_path}. Update DATA_ROOT before continuing.'
        )

print(f'Data root: {DATA_ROOT.resolve()}')
print(f'Figure output: {FIGURE_PATH.resolve()}')

## Functions used to reconstruct the four background-subtracted profiles

The DHS branch follows the recalculated fixed-window implementation used by the original notebook: the end of the logged blank is the reference point (`rp`), with `rp − 30` to `rp` averaged for the blank and `rp + 1` to `rp + 60` for the sample. The DSI branch uses the `start` and `end` values recorded in its log.

For each measurement file, the most intense feature in *m/z* 75.5–76.5 and the most intense feature in *m/z* 133.5–134.5 are identified from their whole-file means. Their summed mean signal is the reagent-ion denominator. Window-averaged target signals are divided by this denominator and multiplied by 10⁶ to obtain normalised counts per second (ncps).


In [ ]:
def extract_mz_from_col_name(column_name):
    """Extract the numerical m/z value from a Tofware column name."""
    marker = 'm/Q '
    marker_position = str(column_name).find(marker)
    if marker_position == -1:
        return None
    try:
        return float(str(column_name)[marker_position + len(marker):].strip("' "))
    except ValueError:
        return None


def find_feature_column(dataframe, target_mz, tolerance=1e-6):
    """Find the mass-feature column matching a specified exact m/z."""
    candidates = []
    for column in dataframe.columns:
        mz_value = extract_mz_from_col_name(column)
        if mz_value is not None:
            candidates.append((abs(mz_value - target_mz), column, mz_value))

    if not candidates:
        raise ValueError('No Tofware mass-feature columns were found.')

    difference, column, observed_mz = min(candidates, key=lambda item: item[0])
    if difference > tolerance:
        raise KeyError(
            f'No feature found for m/z {target_mz:.6f}; nearest was {observed_mz:.6f}.'
        )
    return column


def most_intense_feature_in_range(dataframe, lower_mz, upper_mz):
    """Return the feature with the largest whole-file mean in an m/z interval."""
    candidates = [
        column
        for column in dataframe.columns
        if (mz_value := extract_mz_from_col_name(column)) is not None
        and lower_mz <= mz_value <= upper_mz
    ]
    if not candidates:
        raise KeyError(f'No mass feature found between m/z {lower_mz} and {upper_mz}.')

    means = dataframe[candidates].mean(axis=0)
    selected_column = means.idxmax()
    return selected_column, float(means[selected_column])


def build_normalised_target_table(data_path, mode, target_mzs):
    """Calculate time-window means and reagent-ion-normalised target signals."""
    log = pd.read_csv(data_path / 'log.csv')
    required_log_columns = {
        'sample', 'File_Name', 'hour', 'replicate', 'start', 'end'
    }
    missing_log_columns = required_log_columns.difference(log.columns)
    if missing_log_columns:
        raise KeyError(f'Missing log columns: {sorted(missing_log_columns)}')

    file_cache = {}
    file_information = {}

    for file_name in log['File_Name'].drop_duplicates():
        file_path = data_path / file_name
        if not file_path.exists():
            raise FileNotFoundError(f'Log-referenced measurement file not found: {file_path}')

        dataframe = pd.read_csv(file_path)
        if 't_elapsed_Buf' not in dataframe.columns:
            raise KeyError(f"Column 't_elapsed_Buf' not found in {file_path}")

        target_columns = {
            target_mz: find_feature_column(dataframe, target_mz)
            for target_mz in target_mzs
        }
        mz76_column, mz76_mean = most_intense_feature_in_range(dataframe, 75.5, 76.5)
        mz134_column, mz134_mean = most_intense_feature_in_range(dataframe, 133.5, 134.5)
        reagent_ion_signal = mz76_mean + mz134_mean
        if not np.isfinite(reagent_ion_signal) or reagent_ion_signal <= 0:
            raise ValueError(f'Invalid reagent-ion signal in {file_path}')

        file_cache[file_name] = dataframe
        file_information[file_name] = {
            'target_columns': target_columns,
            'reagent_columns': (mz76_column, mz134_column),
            'reagent_ion_signal': reagent_ion_signal,
        }

    blank_end_by_file = {}
    if mode == 'DHS':
        for file_name in log['File_Name'].drop_duplicates():
            blank_rows = log[
                (log['File_Name'] == file_name)
                & log['sample'].astype(str).str.endswith('_B')
            ]
            if blank_rows.empty:
                raise ValueError(f'No DHS blank entry found for {file_name}')
            blank_end_by_file[file_name] = float(blank_rows.iloc[0]['end'])

    rows = []
    for _, log_row in log.iterrows():
        file_name = log_row['File_Name']
        dataframe = file_cache[file_name]
        information = file_information[file_name]
        is_blank = str(log_row['sample']).endswith('_B')

        if mode == 'DHS':
            reference_point = blank_end_by_file[file_name]
            if is_blank:
                start_time, end_time = reference_point - 30, reference_point
            else:
                start_time, end_time = reference_point + 1, reference_point + 60
        elif mode == 'DSI':
            start_time = float(log_row['start'])
            end_time = float(log_row['end'])
        else:
            raise ValueError("mode must be either 'DHS' or 'DSI'")

        period = dataframe[
            dataframe['t_elapsed_Buf'].between(start_time, end_time, inclusive='both')
        ]
        if period.empty:
            raise ValueError(
                f'Empty {mode} averaging window for {file_name}: {start_time}–{end_time}'
            )

        output_row = {
            'File_Name': file_name,
            'Type': 'Blank' if is_blank else 'Sample',
            'Hour': int(log_row['hour']),
            'Replicate': int(log_row['replicate']),
        }
        for target_mz, target_column in information['target_columns'].items():
            mean_signal = float(period[target_column].mean())
            output_row[f'MZ_{target_mz}'] = (
                mean_signal / information['reagent_ion_signal'] * 1e6
            )
        rows.append(output_row)

    table = pd.DataFrame(rows).sort_values(
        ['Hour', 'Replicate', 'File_Name', 'Type']
    ).reset_index(drop=True)
    return log, table, file_information


def subtract_matched_blanks(normalised_table, target_mzs):
    """Subtract matched normalised blanks and clip negative values to zero."""
    feature_columns = [f'MZ_{target_mz}' for target_mz in target_mzs]
    samples = normalised_table[normalised_table['Type'] == 'Sample'].copy()
    blanks = normalised_table[normalised_table['Type'] == 'Blank'].copy()

    paired = samples.merge(
        blanks,
        on=['File_Name', 'Hour', 'Replicate'],
        suffixes=('_sample', '_blank'),
        validate='one_to_one',
    )

    result = paired[['File_Name', 'Hour', 'Replicate']].copy()
    for feature_column in feature_columns:
        result[feature_column] = np.clip(
            paired[f'{feature_column}_sample'] - paired[f'{feature_column}_blank'],
            a_min=0,
            a_max=None,
        )

    return result.sort_values(['Hour', 'Replicate']).reset_index(drop=True)


## Reconstruct the four replicate-level profiles

This cell reads only the 35 DHS and 24 DSI files referenced by the logs. It generates no intermediate files. The assertions provide compact checks against the supplied supplementary dataset.


In [ ]:
dhs_log, dhs_normalised, dhs_file_information = build_normalised_target_table(
    DATA_ROOT / 'DHS', 'DHS', DHS_TARGETS
)
dsi_log, dsi_normalised, dsi_file_information = build_normalised_target_table(
    DATA_ROOT / 'DSI', 'DSI', DSI_TARGETS
)

dhs_profiles = subtract_matched_blanks(dhs_normalised, DHS_TARGETS)
dsi_profiles = subtract_matched_blanks(dsi_normalised, DSI_TARGETS)

assert len(dhs_file_information) == 35, (
    f'Expected 35 log-referenced DHS files, obtained {len(dhs_file_information)}.'
)
assert len(dsi_file_information) == 24, (
    f'Expected 24 log-referenced DSI files, obtained {len(dsi_file_information)}.'
)
assert len(dhs_profiles) == 35, (
    f'Expected 35 matched DHS sample/blank pairs, obtained {len(dhs_profiles)}.'
)
assert len(dsi_profiles) == 24, (
    f'Expected 24 matched DSI sample/blank pairs, obtained {len(dsi_profiles)}.'
)
assert dhs_profiles['Hour'].nunique() == 12
assert dsi_profiles['Hour'].nunique() == 12

print(f'DHS profiles: {len(dhs_profiles)} replicate observations at '
      f'{dhs_profiles["Hour"].nunique()} oxidation times')
print(f'DSI profiles: {len(dsi_profiles)} replicate observations at '
      f'{dsi_profiles["Hour"].nunique()} oxidation times')


## Prepare the plotting table

The DHS and DSI target columns are reshaped into a common long-format table. Each feature is divided by its maximum value across all replicate-level observations, reproducing the maximum normalisation used in the original figure code.


In [ ]:
dhs_long = dhs_profiles.melt(
    id_vars=['Hour', 'Replicate'],
    value_vars=[f'MZ_{DHS_TARGETS[0]}'],
    var_name='MZ_Peak',
    value_name='Mean_Intensity',
)
dhs_long['MZ_Peak'] = dhs_long['MZ_Peak'] + '_DHS'

dsi_long = dsi_profiles.melt(
    id_vars=['Hour', 'Replicate'],
    value_vars=[f'MZ_{target_mz}' for target_mz in DSI_TARGETS],
    var_name='MZ_Peak',
    value_name='Mean_Intensity',
)
dsi_long['MZ_Peak'] = dsi_long['MZ_Peak'] + '_DSI'

plot_data = pd.concat([dsi_long, dhs_long], ignore_index=True)
plot_data['MZ_Peak'] = pd.Categorical(
    plot_data['MZ_Peak'], categories=PLOT_ORDER, ordered=True
)
plot_data = plot_data.sort_values(['MZ_Peak', 'Hour', 'Replicate']).reset_index(drop=True)

feature_maxima = plot_data.groupby('MZ_Peak', observed=False)['Mean_Intensity'].transform('max')
if (feature_maxima <= 0).any() or feature_maxima.isna().any():
    raise ValueError('At least one selected feature has an invalid maximum intensity.')
plot_data['Normalised_Intensity'] = plot_data['Mean_Intensity'] / feature_maxima

assert plot_data['MZ_Peak'].nunique() == 4
assert len(plot_data) == 107  # 35 DHS observations + 3 × 24 DSI observations
assert np.allclose(
    plot_data.groupby('MZ_Peak', observed=False)['Normalised_Intensity'].max().to_numpy(),
    1.0,
)

summary = (
    plot_data.groupby('MZ_Peak', observed=False)
    .agg(Observations=('Mean_Intensity', 'size'),
         Oxidation_times=('Hour', 'nunique'),
         Maximum_normalised_intensity=('Normalised_Intensity', 'max'))
)
print(summary.to_string())


## Generate and save the figure

The graph reproduces the original square layout, ordering, axis labels, marker style and 95% bootstrap confidence intervals. A fixed bootstrap seed is supplied so that the confidence bands are reproducible across repeated executions with the same software versions.


In [ ]:
import ast
import matplotlib.pyplot as plt
import seaborn as sns

fig, ax = plt.subplots(figsize=(6.732, 6.732))

sns.lineplot(
    data=plot_data,
    x='Hour',
    y='Normalised_Intensity',
    hue='MZ_Peak',
    hue_order=PLOT_ORDER,
    estimator='mean',
    errorbar=('ci', 95),
    n_boot=1000,
    seed=0,
    marker='o',
    markersize=16, # Increased marker size
    ax=ax,
)

ax.set_xlabel('Time (Hours)', fontsize=12)
ax.set_ylabel('Normalised intensity', fontsize=12)
ax.tick_params(axis='both', labelsize=10)
ax.grid(True)

handles, labels = ax.get_legend_handles_labels()
handle_by_label = dict(zip(labels, handles))
ordered_handles = [handle_by_label[label] for label in PLOT_ORDER]

# User can manually override the legend labels here.
# Provide a Python list-like string, e.g., "['Feature A', 'Feature B', 'Feature C', 'Feature D']"
# If left empty, default labels will be used.
manual_legend_labels_input = "['class (i) MZ_344.278_DSI', 'class (ii) MZ_328.283_DSI', 'class (iv) MZ_204.158_DSI', 'class (v) MZ_130.122_DHS']" # @param {type:"string"}

if manual_legend_labels_input:
    try:
        # Attempt to parse the string as a list
        parsed_labels = ast.literal_eval(manual_legend_labels_input)
        if isinstance(parsed_labels, list) and all(isinstance(s, str) for s in parsed_labels):
            if len(parsed_labels) == len(PLOT_ORDER):
                formatted_labels = parsed_labels
            else:
                print(f"Warning: Number of provided labels ({len(parsed_labels)}) does not match number of features ({len(PLOT_ORDER)}). Using default labels.")
                # Fallback to original programmatic generation
                formatted_labels = [
                    f"MZ_{float(label.split('_')[1]):.3f}_{label.split('_')[2]}"
                    for label in PLOT_ORDER
                ]
        else:
            print("Warning: Manual labels input is not a valid list of strings. Using default labels.")
            # Fallback to original programmatic generation
            formatted_labels = [
                f"MZ_{float(label.split('_')[1]):.3f}_{label.split('_')[2]}"
                for label in PLOT_ORDER
            ]
    except (ValueError, SyntaxError) as e:
        print(f"Warning: Error parsing manual legend labels: {e}. Using default labels.")
        # Fallback to original programmatic generation
        formatted_labels = [
            f"MZ_{float(label.split('_')[1]):.3f}_{label.split('_')[2]}"
            for label in PLOT_ORDER
        ]
else:
    # Original programmatic generation
    formatted_labels = [
        f"MZ_{float(label.split('_')[1]):.3f}_{label.split('_')[2]}"
        for label in PLOT_ORDER
    ]

leg = ax.legend(
    ordered_handles,
    formatted_labels,
    title='Anchor features',
    loc='upper center',
    bbox_to_anchor=(0.5, -0.25),
    ncol=2,
    fontsize=10,
    title_fontsize=12,
)

fig.tight_layout()

# Save TIFF version
FIGURE_PATH_TIFF = OUTPUT_ROOT / 'Representative_Time_Profiles_Normalized_to_Max_with_CI.tiff'
fig.savefig(FIGURE_PATH_TIFF, dpi=600, bbox_inches='tight', bbox_extra_artists=[leg])
plt.show()

if not FIGURE_PATH_TIFF.exists() or FIGURE_PATH_TIFF.stat().st_size == 0:
    raise RuntimeError(f'Figure was not created correctly: {FIGURE_PATH_TIFF}')

print(f'Figure saved to: {FIGURE_PATH_TIFF.resolve()}')
print(f'Figure size: {FIGURE_PATH_TIFF.stat().st_size:,} bytes')

# Save JPG version
FIGURE_PATH_JPG = OUTPUT_ROOT / 'Representative_Time_Profiles_Normalized_to_Max_with_CI.jpg'
fig.savefig(FIGURE_PATH_JPG, dpi=300, bbox_inches='tight', bbox_extra_artists=[leg])

if not FIGURE_PATH_JPG.exists() or FIGURE_PATH_JPG.stat().st_size == 0:
    raise RuntimeError(f'JPG figure was not created correctly: {FIGURE_PATH_JPG}')

print(f'JPG figure saved to: {FIGURE_PATH_JPG.resolve()}')
print(f'JPG figure size: {FIGURE_PATH_JPG.stat().st_size:,} bytes')
